# Notebook 4: Preprocessing Clarifications

In this notebook, we focus on key preprocessing steps for hyperscanning analysis:

1. **Union vs. Intersection of Epochs and Channels:**  
   Understand the impact of choosing to keep either all valid epochs/channels (union) or only the epochs/channels common to both participants (intersection).

2. **Artifact Rejection:**  
   Use ICA to remove artifacts (e.g., eye blinks, muscle activity) from EEG data.

3. **Impact on Connectivity Metrics:**  
   Examine how different preprocessing strategies affect the resulting connectivity measures.

4. **Arbitrary Methodological Choices:**  
   Discuss how filtering, baseline correction, and reference choices can influence results and why it's critical to document these decisions.

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from collections import OrderedDict

# HyPyP modules for preprocessing and analysis
import hypyp.prep as prep  # Contains functions for ICA rejection
import hypyp.analyses as analyses  # For synchrony metrics
import hypyp.viz as viz  # For visualization of results

print("Libraries imported successfully.")

## 1. Union vs. Intersection of Channels

When preprocessing data from two participants, you may encounter situations where different channels are marked as bad for each participant.

- **Union of Channels:**  
  Keep all channels that are good in at least one participant. This maximizes spatial coverage but may include less reliable channels for one participant.
  
- **Intersection of Channels:**  
  Only keep channels that are good in both participants. This ensures data quality but may reduce spatial coverage.

Below is an example demonstrating both approaches.

In [ ]:
# Load example epochs (adjust file paths accordingly)
epo1 = mne.read_epochs("./data/participant1-epo.fif", preload=True)
epo2 = mne.read_epochs("./data/participant2-epo.fif", preload=True)

# Using MNE's equalize_epoch_counts ensures we only use epochs that are valid for both participants.
mne.epochs.equalize_epoch_counts([epo1, epo2])

print("Epochs after equalization:")
print("Participant 1:", epo1)
print("Participant 2:", epo2)

In [ ]:
# Suppose the raw montage has the following channels (example):
all_channels = epo1.ch_names  # assume both participants share the same channel list

# Mark bad channels:
epo1.info['bads'] = ['Fp1', 'FT9', 'FC5', 'CP2']  # Participant 1 has FP1, FT9, FC5 and CP2 as bad
epo2.info['bads'] = ['F8', 'Cz', 'CP2']   # Participant 2 has F8, Cz and CP2 as bad

# Compute the "good" channels for each participant (union approach)
good_channels_p1 = [ch for ch in all_channels if ch not in epo1.info['bads']]
good_channels_p2 = [ch for ch in all_channels if ch not in epo2.info['bads']]

print("Union Approach:")
print("Participant 1 good channels:", good_channels_p1)
print("Participant 2 good channels:", good_channels_p2)

# Intersection approach: only keep channels that are good in both participants
common_good_channels = list(set(good_channels_p1).intersection(set(good_channels_p2)))
print("\nIntersection Approach:")
print("Common good channels:", sorted(common_good_channels))

# To apply the intersection approach, restrict each epochs object to the common good channels:
epo1_intersect = epo1.copy().pick(common_good_channels)
epo2_intersect = epo2.copy().pick(common_good_channels)

print("\nAfter applying intersection:")
print("Participant 1 channels:", epo1_intersect.ch_names)
print("Participant 2 channels:", epo2_intersect.ch_names)

## 2. Artifact Rejection: ICA for EEG and Union vs. Intersection Strategies

Independent Component Analysis (ICA) is a powerful technique for removing artifacts such as eye blinks and muscle activity from EEG data. After ICA, additional automatic rejection methods can be applied to further clean the data.

In hyperscanning, we have an important choice: do we reject epochs independently for each participant (union strategy) or do we ensure that the same epochs are kept for both participants (intersection strategy)?

In [ ]:
def create_didactic_visualization():
    """
    Creates a didactic visualization of the difference between union and intersection strategies.
    """
    # Create simulated data
    n_epochs = 10
    # Rejection masks (True = epoch is rejected)
    p1_rejected = np.zeros(n_epochs, dtype=bool)
    p2_rejected = np.zeros(n_epochs, dtype=bool)
    
    # Participant 1 has epochs 2, 5, 8 rejected
    p1_rejected[2] = p1_rejected[5] = p1_rejected[8] = True
    
    # Participant 2 has epochs 3, 6, 9 rejected
    p2_rejected[3] = p2_rejected[6] = p2_rejected[8] = True
    
    # Calculate epochs kept according to each strategy
    p1_kept = ~p1_rejected
    p2_kept = ~p2_rejected
    
    union_kept = p1_kept | p2_kept  # Epochs kept for at least one participant
    intersection_kept = p1_kept & p2_kept  # Epochs kept for both participants
    
    # Create the figure
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    
    # Display the epoch status for each participant (green = kept, red = rejected)
    cmap_rejected = plt.cm.colors.ListedColormap(['green', 'red'])
    
    axes[0].imshow(p1_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[0].set_title('Participant 1 (green = kept, red = rejected)')
    axes[0].set_yticks([])
    
    axes[1].imshow(p2_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[1].set_title('Participant 2 (green = kept, red = rejected)')
    axes[1].set_yticks([])
    
    # Comparison of strategies
    axes[2].imshow(~union_kept.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[2].set_title('Union Strategy (keep if at least one participant has clean data)')
    axes[2].set_yticks([])
    
    axes[3].imshow(~intersection_kept.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[3].set_title('Intersection Strategy (keep only if both participants have clean data)')
    axes[3].set_yticks([])
    
    # Add epoch numbers
    for ax in axes:
        ax.set_xticks(range(n_epochs))
        ax.set_xticklabels(range(1, n_epochs+1))
        ax.grid(True, axis='x', linestyle='--', alpha=0.7)
    
    plt.xlabel('Epoch Number')
    
    # Summary of totals
    plt.figtext(0.5, 0.01, 
                f"Epochs kept - Union: {sum(union_kept)}/{n_epochs}, Intersection: {sum(intersection_kept)}/{n_epochs}",
                ha='center', fontsize=12, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.8))
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust for the summary text
    plt.show()

# Call the function to create the didactic visualization
create_didactic_visualization()

In [ ]:
# Clean bad channels for ICA processing
epo1.info['bads'] = []
epo2.info['bads'] = []

# Introduce artifacts to better show differences
def introduce_artifacts(epo1, epo2):
    """
    Artificially introduces artifacts into the EEG data to illustrate
    the difference between union and intersection strategies.
    """
    # Create copies of the Epochs objects to avoid modifying the originals
    epo1_mod = epo1.copy()
    epo2_mod = epo2.copy()
    
    n_epochs = len(epo1)
    n_channels = len(epo1.ch_names)
    
    # Define specific epochs to disturb for each participant
    # For better illustration, we want different rejection patterns
    artifact_epochs_p1 = [0, 2, 4, 6, 8]  # Participant 1: disturb even epochs
    artifact_epochs_p2 = [1, 2, 5, 7, 8]  # Participant 2: disturb odd epochs
    
    # For each participant, introduce different types of artifacts
    
    # 1. Participant 1: Add high amplitude artifacts (like eye blinks)
    for epoch_idx in artifact_epochs_p1:
        if epoch_idx < n_epochs:
            # Select frontal channels (typically affected by eye blinks)
            frontal_channels = [i for i, ch in enumerate(epo1.ch_names) 
                               if ch.startswith(('Fp', 'F')) and i < n_channels]
            
            if frontal_channels:
                # Add an artificial "blink"
                for ch_idx in frontal_channels[:3]:  # Limit to a few frontal channels
                    # Create a waveform resembling a blink
                    peak_time = np.random.randint(100, epo1.times.size - 100)
                    artifact = np.zeros(epo1.times.size)
                    artifact[peak_time-50:peak_time+50] = 200 * np.exp(-0.1 * np.abs(np.arange(-50, 50)))
                    
                    # Add the artifact to the data
                    epo1_mod._data[epoch_idx, ch_idx, :] += artifact
    
    # 2. Participant 2: Add high frequency noise (like muscle artifacts)
    for epoch_idx in artifact_epochs_p2:
        if epoch_idx < n_epochs:
            # Select temporal channels (typically affected by muscle artifacts)
            temporal_channels = [i for i, ch in enumerate(epo2.ch_names) 
                                if ch.startswith(('T', 'C')) and i < n_channels]
            
            if temporal_channels:
                # Add artificial "muscle noise"
                for ch_idx in temporal_channels[:3]:  # Limit to a few temporal channels
                    # Create high frequency noise
                    high_freq_noise = np.random.normal(0, 50, epo2.times.size)
                    # Slightly smooth the noise to make it more realistic
                    from scipy.ndimage import gaussian_filter1d
                    high_freq_noise = gaussian_filter1d(high_freq_noise, sigma=1)
                    
                    # Apply the noise to only a portion of the epoch
                    start_idx = np.random.randint(0, epo2.times.size // 2)
                    end_idx = start_idx + epo2.times.size // 3  # Duration of the artifact
                    end_idx = min(end_idx, epo2.times.size)  # Make sure not to exceed
                    
                    # Add the artifact to the data
                    epo2_mod._data[epoch_idx, ch_idx, start_idx:end_idx] += high_freq_noise[start_idx:end_idx]
    
    print(f"Artifacts introduced - Participant 1: epochs {artifact_epochs_p1}")
    print(f"Artifacts introduced - Participant 2: epochs {artifact_epochs_p2}")
    
    return epo1_mod, epo2_mod, artifact_epochs_p1, artifact_epochs_p2

In [ ]:
# Apply the modification
epo1_mod, epo2_mod, artifact_p1, artifact_p2 = introduce_artifacts(epo1, epo2)

# Apply ICA on the modified data
print("Applying ICA on modified data...")
icas_mod = prep.ICA_fit([
    epo1_mod, epo2_mod
],
    n_components=15,
    method='infomax',
    fit_params=dict(extended=True),
    random_state=42
)

cleaned_epochs_ICA_mod = prep.ICA_choice_comp(icas_mod, [epo1_mod, epo2_mod])
print('ICA correction completed.')

In [ ]:
# Apply AutoReject with UNION strategy
print("Applying AutoReject with UNION strategy...")
cleaned_epochs_AR_union, dic_AR_union = prep.AR_local(
    cleaned_epochs_ICA_mod,
    strategy="union",
    threshold=50.0,
    verbose=True
)

# Apply AutoReject with INTERSECTION strategy
print("Applying AutoReject with INTERSECTION strategy...")
cleaned_epochs_AR_intersection, dic_AR_intersection = prep.AR_local(
    cleaned_epochs_ICA_mod,
    strategy="intersection",
    threshold=50.0,
    verbose=True
)

# Assign cleaned epochs
epo1_clean_union = cleaned_epochs_AR_union[0]
epo2_clean_union = cleaned_epochs_AR_union[1]
epo1_clean_intersection = cleaned_epochs_AR_intersection[0]
epo2_clean_intersection = cleaned_epochs_AR_intersection[1]

print(f'Number of epochs with UNION strategy - Participant 1: {len(epo1_clean_union)}, Participant 2: {len(epo2_clean_union)}')
print(f'Number of epochs with INTERSECTION strategy - Participant 1: {len(epo1_clean_intersection)}, Participant 2: {len(epo2_clean_intersection)}')

In [ ]:
def visualize_clean_epochs_comparison(epo1_clean_union, epo2_clean_union, 
                                     epo1_clean_intersection, epo2_clean_intersection):
    """
    Visualize the differences in epoch counts and data quality between union and intersection strategies.
    """
    # Get counts
    n_union_1 = len(epo1_clean_union)
    n_union_2 = len(epo2_clean_union)
    n_intersect_1 = len(epo1_clean_intersection)
    n_intersect_2 = len(epo2_clean_intersection)
    
    # Print summary
    print("Summary of Epoch Counts:")
    print(f"Union strategy - Participant 1: {n_union_1} epochs")
    print(f"Union strategy - Participant 2: {n_union_2} epochs")
    print(f"Intersection strategy - Participant 1: {n_intersect_1} epochs")
    print(f"Intersection strategy - Participant 2: {n_intersect_2} epochs")
    
    # Create a bar chart
    fig, ax = plt.subplots(figsize=(10, 6))
    
    strategies = ['Union', 'Intersection']
    p1_counts = [n_union_1, n_intersect_1]
    p2_counts = [n_union_2, n_intersect_2]
    
    x = np.arange(len(strategies))
    width = 0.35
    
    ax.bar(x - width/2, p1_counts, width, label='Participant 1')
    ax.bar(x + width/2, p2_counts, width, label='Participant 2')
    
    ax.set_xticks(x)
    ax.set_xticklabels(strategies)
    ax.set_ylabel('Number of Epochs')
    ax.set_title('Comparison of Clean Epochs: Union vs. Intersection')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Plot a sample of the data from each participant and strategy
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    sample_epoch = 0  # Choose the first epoch (adjust if needed)
    
    # If there are no epochs, skip plotting
    if n_union_1 > 0:
        plot_epoch_data(epo1_clean_union, sample_epoch, axes[0, 0], 'Union - Participant 1')
    else:
        axes[0, 0].text(0.5, 0.5, 'No epochs available', ha='center')
        axes[0, 0].set_title('Union - Participant 1')
    
    if n_union_2 > 0:
        plot_epoch_data(epo2_clean_union, sample_epoch, axes[0, 1], 'Union - Participant 2')
    else:
        axes[0, 1].text(0.5, 0.5, 'No epochs available', ha='center')
        axes[0, 1].set_title('Union - Participant 2')
    
    if n_intersect_1 > 0:
        plot_epoch_data(epo1_clean_intersection, sample_epoch, axes[1, 0], 'Intersection - Participant 1')
    else:
        axes[1, 0].text(0.5, 0.5, 'No epochs available', ha='center')
        axes[1, 0].set_title('Intersection - Participant 1')
    
    if n_intersect_2 > 0:
        plot_epoch_data(epo2_clean_intersection, sample_epoch, axes[1, 1], 'Intersection - Participant 2')
    else:
        axes[1, 1].text(0.5, 0.5, 'No epochs available', ha='center')
        axes[1, 1].set_title('Intersection - Participant 2')
    
    plt.tight_layout()
    plt.show()

def plot_epoch_data(epochs, epoch_idx, ax, title):
    """
    Plot a single epoch's data on the given axis.
    """
    if epoch_idx >= len(epochs):
        ax.text(0.5, 0.5, f'Epoch {epoch_idx} not available', ha='center')
        ax.set_title(title)
        return
    
    # Get data for the specified epoch
    data = epochs.get_data(copy=True)[epoch_idx]
    times = epochs.times
    
    # Plot each channel (limit to 10 channels for clarity)
    max_channels = min(10, data.shape[0])
    for i in range(max_channels):
        ax.plot(times, data[i], alpha=0.7, label=f'Ch {i+1}' if i < 5 else '')
    
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    
    # Only show legend for the first 5 channels to avoid clutter
    if max_channels > 0:
        ax.legend(loc='upper right', fontsize='small')

# Call the new visualization function
visualize_clean_epochs_comparison(epo1_clean_union, epo2_clean_union, 
                                 epo1_clean_intersection, epo2_clean_intersection)

## 3. Impact on Connectivity Metrics

To understand how the choice of preprocessing strategy affects our results, we'll compute connectivity metrics for both the union and intersection approaches and compare them.

In [ ]:
# Define frequency bands for analysis
freq_bands = OrderedDict({
    'Alpha-Low': [7.5, 11],
    'Alpha-High': [11.5, 13]
})

# Extract sampling rate
sampling_rate = epo1.info['sfreq']

# Function to compute connectivity for a set of epochs
def compute_connectivity(epo1, epo2, freq_bands, sampling_rate):
    """
    Computes connectivity metrics between two sets of epochs.
    """
    # Prepare data for connectivity analysis
    dyad_data = np.array([epo1.get_data(copy=True), epo2.get_data(copy=True)])
    
    # Compute analytic signal
    complex_signal = analyses.compute_freq_bands(
        dyad_data, 
        sampling_rate, 
        freq_bands,
        filter_length=int(sampling_rate),
        l_trans_bandwidth=5.0,
        h_trans_bandwidth=5.0
    )
    
    # Compute connectivity using circular correlation
    result = analyses.compute_sync(complex_signal, mode='ccorr', epochs_average=True)
    
    # Get number of channels
    n_ch = len(epo1.info['ch_names'])
    
    # Extract inter-brain connectivity (alpha low band)
    alpha_low = result[0, 0:n_ch, n_ch:2*n_ch]
    
    return alpha_low

# Compute connectivity for both strategies
print("Computing connectivity metrics for UNION strategy...")
connectivity_union = compute_connectivity(epo1_clean_union, epo2_clean_union, freq_bands, sampling_rate)

print("Computing connectivity metrics for INTERSECTION strategy...")
connectivity_intersection = compute_connectivity(epo1_clean_intersection, epo2_clean_intersection, freq_bands, sampling_rate)

# Compare the connectivity matrices
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
im1 = plt.imshow(connectivity_union, cmap='viridis')
plt.colorbar(im1, label='Connectivity (CCORR)')
plt.title('Alpha-Low Connectivity - UNION Strategy')
plt.xlabel('Participant 2 Channels')
plt.ylabel('Participant 1 Channels')

plt.subplot(1, 2, 2)
im2 = plt.imshow(connectivity_intersection, cmap='viridis')
plt.colorbar(im2, label='Connectivity (CCORR)')
plt.title('Alpha-Low Connectivity - INTERSECTION Strategy')
plt.xlabel('Participant 2 Channels')
plt.ylabel('Participant 1 Channels')

plt.tight_layout()
plt.show()

# Also calculate the difference between the two to highlight discrepancies
plt.figure(figsize=(8, 6))
diff = connectivity_union - connectivity_intersection
im3 = plt.imshow(diff, cmap='coolwarm')
plt.colorbar(im3, label='Connectivity Difference (Union - Intersection)')
plt.title('Difference in Connectivity (Union - Intersection)')
plt.xlabel('Participant 2 Channels')
plt.ylabel('Participant 1 Channels')
plt.tight_layout()
plt.show()

# Summary statistics of differences
print("Summary of Connectivity Differences:")
print(f"Mean absolute difference: {np.mean(np.abs(diff)):.4f}")
print(f"Maximum absolute difference: {np.max(np.abs(diff)):.4f}")
print(f"Percentage of channels with differences > 0.05: {np.mean(np.abs(diff) > 0.05) * 100:.1f}%")

## 4. Arbitrary Methodological Choices

Preprocessing involves several choices that can affect final results. Key decisions include:
- **Filtering:**  
  The selection of lower and upper frequency cutoffs.  
  Example: A bandpass filter from 1 Hz to 40 Hz might be used for EEG.
- **Baseline Correction:**  
  Choosing the time window for baseline correction.
- **Reference Selection:**  
  For EEG, choosing a common average, linked mastoids, or another reference.
- **Epoch Rejection Criteria:**  
  Deciding on thresholds for rejecting noisy epochs.

Below is an example that demonstrates filtering and baseline correction.

In [ ]:
# Apply a bandpass filter and baseline correction to the cleaned epochs.
epo1_filtered = epo1_clean_intersection.copy().filter(l_freq=1., h_freq=40.)
epo2_filtered = epo2_clean_intersection.copy().filter(l_freq=1., h_freq=40.)

# Apply baseline correction (e.g., from -0.2 to 0 seconds)
epo1_filtered.apply_baseline((None, 0))
epo2_filtered.apply_baseline((None, 0))

print("Filtering and baseline correction applied to EEG data.")

# Visualize a few epochs after preprocessing
epo1_filtered.plot(n_epochs=5, n_channels=8, title="Preprocessed EEG Data - Participant 1")

## Discussion

- **Union vs. Intersection in Channel Selection:**  
  The choice between union and intersection approaches affects spatial coverage. The intersection approach ensures that all included channels have reliable data from both participants but may reduce spatial coverage.

- **Union vs. Intersection in Epoch Rejection:**  
  The choice between union and intersection strategies for epoch rejection has significant implications:
  
  - **Union strategy** keeps epochs that are valid in at least one participant. This maximizes the amount of data retained but may include epochs where one participant's data is less reliable.
  
  - **Intersection strategy** only keeps epochs that are valid in both participants. This ensures that all included epochs have high-quality data from both participants, which is crucial for hyperscanning analysis, but typically results in fewer epochs.

  Our comparison demonstrates that these choices can significantly impact the resulting connectivity metrics. The differences are particularly notable when participants have different artifact patterns, which is common in real-world hyperscanning recordings.

- **Impact on Connectivity Metrics:**  
  The connectivity matrices show that the choice of epoch rejection strategy affects the estimated inter-brain synchrony. The intersection approach typically produces more conservative (and potentially more reliable) synchrony estimates, as it only includes high-quality data from both participants.

- **Methodological Choices:**  
  Filtering, baseline correction, and reference selection are often chosen arbitrarily, but they can greatly influence the analysis outcomes. It is critical to document these choices for reproducibility.

By clearly understanding and documenting these preprocessing steps, we can ensure that the results of our hyperscanning analysis are both robust and interpretable.

## Conclusion

In this notebook, we have:
- Illustrated the difference between union and intersection strategies in both channel selection and epoch rejection.
- Demonstrated how these different rejection strategies impact the resulting connectivity metrics.
- Demonstrated artifact rejection using ICA for EEG data.
- Discussed several arbitrary methodological choices (filtering, baseline correction, etc.) that impact the final analysis.

These preprocessing clarifications are crucial for obtaining reliable inter-brain synchrony metrics and ensuring reproducibility in hyperscanning studies. When designing a hyperscanning study, researchers should carefully consider which preprocessing strategy best aligns with their research questions and experimental design.